# CASALS L1B data and official `refh`

Each pulse contributes one official geolocated `refh` point. Waveform components are diagnostics and do not become extra geolocated returns. `refh` is interpreted as WGS84 ellipsoidal height unless source metadata documents otherwise; horizontal reprojection does not change that vertical reference.

The example below checks a bounded H5 schema summary. Full `refh` loading is opt-in because it reads several arrays for every pulse.

In [1]:
from pathlib import Path

H5_PATH = Path("data/raw/casals_l1b/casals_l1b_20241112T165718_001_02.h5")
READ_ALL_REFH = False  # Set True only when a full point-array read is intended.

In [2]:
import h5py

from casals_l1b.h5 import read_global_attributes, find_dataset

if H5_PATH.is_file():
    with h5py.File(H5_PATH, "r") as h5:
        attrs = read_global_attributes(h5)
        print({key: attrs[key] for key in ("n_pulses", "n_sweeps", "n_tracks", "n_rx_bins") if key in attrs})
        for name in ("refh_longitude", "refh_latitude", "refh", "refh_amp", "refh_snr", "rx_waveform"):
            ds = find_dataset(h5, name)
            print(name, None if ds is None else {"path": ds.name, "shape": ds.shape, "dtype": str(ds.dtype)})
else:
    print(f"Set H5_PATH to a local CASALS L1B file; not found: {H5_PATH}")

{'n_pulses': 3604480, 'n_sweeps': 14080, 'n_tracks': 256, 'n_rx_bins': 2728}
refh_longitude {'path': '/refh_longitude', 'shape': (3604480,), 'dtype': 'float64'}
refh_latitude {'path': '/refh_latitude', 'shape': (3604480,), 'dtype': 'float64'}
refh {'path': '/refh', 'shape': (3604480,), 'dtype': 'float64'}
refh_amp {'path': '/refh_amp', 'shape': (3604480,), 'dtype': 'int16'}
refh_snr {'path': '/refh_snr', 'shape': (3604480,), 'dtype': 'float64'}
rx_waveform {'path': '/rx_waveform', 'shape': (3604480, 2728), 'dtype': 'int16'}


In [3]:
if READ_ALL_REFH and H5_PATH.is_file():
    from casals_l1b.refh import read_refh_points, project_refh_points

    points = read_refh_points(H5_PATH)
    projected = project_refh_points(points)
    print(points.n_input_records, points.n_valid_records, projected.utm_epsg)
    print(points.input_mask_summary)
else:
    print("Full refh read skipped. Set READ_ALL_REFH=True after reviewing file size and available memory.")

Full refh read skipped. Set READ_ALL_REFH=True after reviewing file size and available memory.
